# Qwen3-0.6B Turkish LoRA pilot

This notebook downloads the training code from `dozagoat47-star/qwen3-turkish-lora-kaggle` and reads the reviewed dialogue JSONL from a separate **private** Kaggle Dataset. The dialogue data is not approved for public redistribution and must not be committed to GitHub. Before running, create a private Kaggle Dataset containing `qwen3_dialogue_review_001.jsonl`, attach it here, enable a GPU accelerator, and enable internet access.

The dataset has only 20 conversations. Seed 7 keeps each conversation intact: 16 train, 2 validation, 2 held out for one final qualitative comparison. This pilot cannot establish general conversational quality. Do not use the held-out comparison to tune this run.

The adapter, training metadata, and held-out report are written under `/kaggle/working/qwen3_0_6b_lora_run_001`; save/commit the Kaggle notebook version and download those output files after it completes. If setup already ran in this session, rerun the setup cell to fetch the latest code before retrying training.

In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = 'https://github.com/dozagoat47-star/qwen3-turkish-lora-kaggle.git'
GITHUB_REF = 'main'

repo_root = Path('/kaggle/working/qwen3_public_code')
if repo_root.exists():
    if not (repo_root / '.git').is_dir():
        raise FileExistsError(f'{repo_root} exists but is not the expected Git clone')
    subprocess.check_call(['git', '-C', str(repo_root), 'fetch', '--depth', '1', 'origin', GITHUB_REF])
    subprocess.check_call(['git', '-C', str(repo_root), 'checkout', '--detach', 'FETCH_HEAD'])
else:
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', GITHUB_REF,
        GITHUB_REPO_URL, str(repo_root),
    ])
required_code = [
    repo_root / 'clean_model' / '__init__.py',
    repo_root / 'clean_model' / 'data.py',
    repo_root / 'clean_model' / 'qwen_lora.py',
]
missing_code = [str(path.relative_to(repo_root)) for path in required_code if not path.is_file()]
if missing_code:
    raise FileNotFoundError(f'GitHub repository is missing required code files: {missing_code}')
github_commit = subprocess.check_output(
    ['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True
).strip()
for name in ('train_adapter', 'evaluate_test_once'):
    globals().pop(name, None)
for module_name in tuple(sys.modules):
    if module_name == 'clean_model' or module_name.startswith('clean_model.'):
        del sys.modules[module_name]
sys.path.insert(0, str(repo_root))

input_root = Path('/kaggle/input')
data_files = list(input_root.rglob('qwen3_dialogue_review_001.jsonl'))
if len(data_files) != 1:
    raise RuntimeError(
        f'Expected exactly one attached qwen3_dialogue_review_001.jsonl; found {len(data_files)}. '
        'Attach a private Kaggle Dataset containing the approved JSONL file.'
    )

data_path = data_files[0]
expected_data_sha256 = '622f6e9d093cfa81517eed2b1d6767a0d619a58048a4ce80ee51091256d4b2a1'
expected_fingerprint = 'ef86621b783db3644a6518de31327a9d22fbb5a2c2257813f4d340489ee0dd09'
data_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
if data_sha256 != expected_data_sha256:
    raise ValueError(f'Private dataset SHA-256 mismatch: {data_sha256}')
from clean_model.data import dataset_fingerprint, load_jsonl

records = load_jsonl(data_path, require_approved=True)
if dataset_fingerprint(records) != expected_fingerprint:
    raise ValueError('Canonical dataset fingerprint does not match the approved pilot data')
if len(records) != 20 or any(record['kind'] != 'dialogue' for record in records):
    raise ValueError('Expected exactly 20 approved dialogue records')
if any(record['review']['reviewed_by'] != 'project_owner' for record in records):
    raise ValueError('All records must be reviewed by the project owner')
print(f'Public code commit: {github_commit}')
print(f'Private dataset verified: {len(records)} approved conversations')
print(f'Dataset fingerprint: {expected_fingerprint}')

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'transformers>=5,<6', 'peft>=0.17,<1', 'accelerate>=1,<2',
    'huggingface-hub>=1.16,<2',
])

## Pilot configuration

The fixed model revision and deterministic group split are recorded in the training metadata. The small validation set is used only to select the best epoch; the test split is not opened during training.

In [ ]:
from clean_model.qwen_lora import train_adapter

run_dir = Path('/kaggle/working/qwen3_0_6b_lora_run_001')
training_metadata = train_adapter(
    dataset_path=data_path,
    output_dir=run_dir,
    epochs=8,
    batch_size=2,
    learning_rate=2e-4,
    patience=3,
    seed=7,
)
print(json.dumps(training_metadata, ensure_ascii=False, indent=2))

## One-shot held-out comparison

Run this cell once only after training and hyperparameters are fixed. It compares greedy generations from the unchanged base checkpoint and the selected adapter on the two held-out conversations, then saves the complete report. The tiny test set is for qualitative inspection, not a reliable score.

In [ ]:
import gc
import torch

evaluation_lock = run_dir / 'test_evaluation_started.lock'
with evaluation_lock.open('x', encoding='utf-8') as lock:
    lock.write('Held-out test evaluation started once. Do not rerun for tuning.\n')
gc.collect()
torch.cuda.empty_cache()
from clean_model.qwen_lora import evaluate_test_once

test_report = evaluate_test_once(
    dataset_path=data_path,
    adapter_dir=run_dir,
    report_path=run_dir / 'one_shot_test_evaluation.json',
    seed=7,
)
print(json.dumps(test_report, ensure_ascii=False, indent=2))